# **_----Baiseline для моделей DL---_**

In [ ]:
"""""Модуль с архитектурой нейронной сети
Реализована на PyTorch в виде полносвязной сети (MLP) с BatchNorm и Dropout.
"""""

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import config
from preprocessing import preprocess_data
# Фиксируем случайность везде, чтобы результаты PyTorch повторялись точь-в-точь:
np.random.seed(config.RANDOM_STATE)
torch.manual_seed(config.RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.RANDOM_STATE)
# Проверяем доступность видеокарты (если нет — считаем на процессоре CPU):
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Обучение будет на устройстве: {device}")

Обучение будет на устройстве: cpu


In [2]:
# Загрузка данных
df_train = pd.read_csv(config.TRAIN_PATH)
df_test = pd.read_csv(config.TEST_PATH)
test_ids = df_test[config.ID_COL]


# Предобработка
df_train_proc, df_test_proc = preprocess_data(df_train, df_test)
X_train = df_train_proc.drop(columns=[config.ID_COL, config.TARGET_COL])
y_train = np.log1p(df_train_proc[config.TARGET_COL])
X_test = df_test_proc.drop(columns=[config.ID_COL])
kf = KFold(n_splits=config.N_SPLITS, shuffle=config.SHUFFLE, random_state=config.RANDOM_STATE)
print(f"Данные готовы! X_train: {X_train.shape}, y_train: {y_train.shape}")

Данные готовы! X_train: (1460, 208), y_train: (1460,)


In [9]:
import importlib
importlib.reload(config)

class HouseMLP(nn.Module):
    """
    Полносвязная нейросеть для регрессии стоимости домов:
    - Слой 1: Linear -> BatchNorm -> ReLU -> Dropout
    - Слой 2: Linear -> BatchNorm -> ReLU -> Dropout
    - Выход: 1 непрерывное число (прогноз логарифма цены)
    """
    def __init__(self, input_dim: int, hidden_dim: int = config.DL_CONFIG["hidden_dim"], dropout: float = config.DL_CONFIG["dropout"]):
        super(HouseMLP, self).__init__()
        # 1-й блок: преобразует 208 признаков в hidden_dim
        self.block1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        # 2-й блок: сжимает с hidden_dim до hidden_dim // 2
        self.block2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        # Выходной слой: выдает 1 число
        self.out_layer = nn.Linear(hidden_dim // 2, 1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Прямой проход данных через слои"""
        x = self.block1(x)
        x = self.block2(x)
        x = self.out_layer(x)
        return x

In [14]:
# =============================================================
# ТРЕНИРОВОЧНЫЙ ЦИКЛ PYTORCH (5-FOLD CV)
# =============================================================


# Массивы для сбора результатов
dl_fold_scores = []
oof_dl_predictions = np.zeros(len(X_train))

# Считываем гиперпараметры из config.DL_CONFIG:
EPOCHS = config.DL_CONFIG["epochs"]
BATCH_SIZE = config.DL_CONFIG["batch_size"]
LR = config.DL_CONFIG["lr"]

print(f"Запуск обучения HouseMLP (Эпох: {EPOCHS}, Батч: {BATCH_SIZE}, LR: {LR})...\n")

# ВНЕШНИЙ ЦИКЛ: 5 фолдов кросс-валидации
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):

    # 1Разделение данных текущего фолда
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    # Масштабирование признаков (StandardScaler строго внутри фолда!)
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)

    # Масштабируем таргет y (приводим логарифмы к среднему 0 и std 1):
    scaler_y = StandardScaler()
    y_tr_scaled = scaler_y.fit_transform(y_tr.values.reshape(-1, 1))

     # Перевод данных в Тензоры PyTorch и перенос на устройство (CPU / GPU)
    train_x_tensor = torch.tensor(X_tr_scaled, dtype=torch.float32).to(device)
    train_y_tensor = torch.tensor(y_tr_scaled, dtype=torch.float32).to(device)
    val_x_tensor = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)

    # Упаковка в DataLoader (нарезка на батчи по 32 штуки)
    train_dataset = TensorDataset(train_x_tensor, train_y_tensor)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    # Создаем СВЕЖИЙ экземпляр модели, лосс и оптимизатор для этого фолда:
    model = HouseMLP(input_dim=X_train.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    # СРЕДНИЙ ЦИКЛ: 60 эпох обучения
    for epoch in range(1, EPOCHS + 1):
        model.train()  # Включаем режим обучения (активирует Dropout и BatchNorm)

        # ВНУТРЕННИЙ ЦИКЛ: проход по батчам
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()               # 1. Сброс старых градиентов
            predictions = model(batch_x)        # 2. Прямой проход (прогноз)
            loss = criterion(predictions, batch_y) # 3. Расчет ошибки MSE
            loss.backward()                     # 4. Обратный проход (считаем градиенты)
            optimizer.step()                    # 5. Шаг оптимизатора (обновляем веса)

    # ВАЛИДАЦИЯ ФОЛДА (когда все 60 эпох завершены):
    model.eval()  # Отключаем Dropout
    with torch.no_grad():  # Отключаем расчет градиентов для экономии памяти
        val_preds_scaled = model(val_x_tensor).cpu().numpy()
        # Переводим прогноз из тензора PyTorch обратно в массив NumPy:
        val_preds = scaler_y.inverse_transform(val_preds_scaled).flatten()

    # Фиксируем результат фолда
    oof_dl_predictions[val_idx] = val_preds
    fold_rmsle = root_mean_squared_error(y_val, val_preds)
    dl_fold_scores.append(fold_rmsle)

    print(f"Фолд {fold} | RMSLE: {fold_rmsle:.4f}")


# =============================================================
# ИТОГОВЫЕ МЕТРИКИ НЕЙРОСЕТИ
# =============================================================


mean_dl_rmsle = np.mean(dl_fold_scores)
std_dl_rmsle = np.std(dl_fold_scores)
real_dollars = np.expm1(y_train)
pred_dollars = np.expm1(oof_dl_predictions)
dl_mae = mean_absolute_error(real_dollars, pred_dollars)
dl_r2 = r2_score(y_train, oof_dl_predictions)


print("=" * 50)
print(f"Итоговый средний RMSLE (PyTorch MLP): {mean_dl_rmsle:.4f} (+/- {std_dl_rmsle:.4f})")
print(f"Средняя ошибка (MAE) в долларах:     ${dl_mae:,.2f}")
print(f"Коэффициент детерминации (R^2):       {dl_r2:.4f}")
print("=" * 50)

Запуск обучения HouseMLP (Эпох: 60, Батч: 32, LR: 0.001)...

Фолд 1 | RMSLE: 0.1526
Фолд 2 | RMSLE: 0.1478
Фолд 3 | RMSLE: 0.1985
Фолд 4 | RMSLE: 0.1415
Фолд 5 | RMSLE: 0.1300
Итоговый средний RMSLE (PyTorch MLP): 0.1541 (+/- 0.0235)
Средняя ошибка (MAE) в долларах:     $18,137.28
Коэффициент детерминации (R^2):       0.8477
